
# Assembling and Evaluating a RAG Application

In the previous demo, we created a Vector Search Index. To build a complete RAG application, it is time to connect all the components that you have learned so far and evaluate the performance of the RAG.

After evaluating the performance of the RAG pipeline, we will create and deploy a new Model Serving Endpoint to perform RAG.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

- Describe embeddings, vector databases, and search/retrieval as key components of implementing performant RAG applications.
- Assemble a RAG pipeline by combining various components.
- Build a RAG evaluation pipeline with MLflow evaluation functions.
- Register a RAG pipeline to the Model Registry.


## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**

In [0]:
%pip install -U -qq databricks-vectorsearch langchain==0.3.7 flashrank langchain-databricks PyPDF2
dbutils.library.restartPython()

## Demo Overview

As seen in the diagram below, in this demo we will focus on the inference section (highlighted in green). The main focus of the previous demos was  Step 1 - Data preparation and vector storage. Now, it is time put all components together to create a RAG application. 

The flow will be the following:

- A user asks a question
- The question is sent to our serverless Chatbot RAG endpoint
- The endpoint compute the embeddings and searches for docs similar to the question, leveraging the Vector Search Index
- The endpoint creates a prompt enriched with the doc
- The prompt is sent to the Foundation Model Serving Endpoint
- We display the output to our users!


<img src="https://files.training.databricks.com/images/genai/genai-as-01-llm-rag-self-managed-flow-2.png" width="100%">




## Setup the RAG Components

In this section, we will first define the components that we created before. Next, we will set up the retriever component for the application. Then, we will combine all the components together. In the final step, we will register the developed application as a model in the Model Registry with Unity Catalog.

### Setup the Retriever

We will setup the Vector Search endpoint that we created in the previous demos as retriever. The retriever will return 2 relevant documents based on the query.


In [0]:
# components we created before
# assign vs search endpoint by username
vs_endpoint_prefix = "vs_endpoint_"

vs_endpoint_name = vs_endpoint_prefix + "stories"
print(f"Assigned Vector Search endpoint name: {vs_endpoint_name}.")

vs_index_fullname = "training_catalog.training_schema.pdf_text_self_managed_vs_index"

In [0]:
from databricks.vector_search.client import VectorSearchClient
from langchain_databricks import DatabricksEmbeddings
from langchain_core.runnables import RunnableLambda
from langchain.docstore.document import Document
from flashrank import Ranker, RerankRequest

def get_retriever(cache_dir="/tmp"):

    def retrieve(query, k: int=20):
        if isinstance(query, dict):
            query = next(iter(query.values()))

        # get the vector search index
        vsc = VectorSearchClient(disable_notice=True)
        vs_index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=vs_index_fullname)
        
        # get the query vector
        embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")
        query_vector = embeddings.embed_query(query)
        
        # get similar k documents
        return query, vs_index.similarity_search(
            query_vector=query_vector,
            columns=["pdf_name", "content"],
            num_results=k)

    def rerank(query, retrieved, cache_dir, k: int=10):
        # format result to align with reranker lib format 
        passages = []
        for doc in retrieved.get("result", {}).get("data_array", []):
            new_doc = {"file": doc[0], "text": doc[1]}
            passages.append(new_doc)       
        # Load the flashrank ranker
        ranker = Ranker(model_name="rank-T5-flan", cache_dir=cache_dir)

        # rerank the retrieved documents
        rerankrequest = RerankRequest(query=query, passages=passages)
        results = ranker.rerank(rerankrequest)[:k]

        # format the results of rerank to be ready for prompt
        return [Document(page_content=r.get("text"), metadata={"source": r.get("file")}) for r in results]

    # the retriever is a runnable sequence of retrieving and reranking.
    return RunnableLambda(retrieve) | RunnableLambda(lambda x: rerank(x[0], x[1], cache_dir))

# test our retriever
question = {"input": "Animal story where two goats fought and drowned."}
vectorstore = get_retriever(cache_dir="/tmp/opt")
similar_documents = vectorstore.invoke(question)
print(f"Relevant documents: {similar_documents}")

### Setup the Foundation Model

Our chatbot will be using `llama3.1` foundation model to provide answer. 

While the model is available using the built-in [Foundation endpoint](/ml/endpoints), we can use Databricks Langchain Chat Model wrapper to easily build our chain.  

Note: multiple type of endpoint or langchain models can be used.

- Databricks Foundation models (what we'll use)
- Your fined-tune model
- An external model provider (such as Azure OpenAI)

In [0]:
from langchain_databricks import ChatDatabricks

# test Databricks Foundation LLM model
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens = 300)
print(f"Test chat model: {chat_model.invoke('What is Generative AI?')}")

## Assembling the Complete RAG Solution

Let's now merge the retriever and the model in a single Langchain chain.

We will use a custom langchain template for our assistant to give proper answer.

Make sure you take some time to try different templates and adjust your assistant tone and personality for your requirement.

<img src="https://files.training.databricks.com/images/genai/genai-as-01-llm-rag-self-managed-model-2.png" width="100%" />

Some important notes about the LangChain formatting:

* Context documents retrieved from the vector store are added by separated newline.

In [0]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# Step 1: Create the new prompt
TEMPLATE = """You are an assistant helping to answer questions based on short animal stories. 
Each story teaches a moral lesson. 
Answer the question based only on the provided stories. 
If the question is not related to the stories, politely say "I don't know based on the given information." 
Do not try to invent or guess answers.
Keep the answer concise and to the point.

Use the following pieces of context to answer the question:

<context>
{context}
</context>

Question: {input}

Answer:
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "input"])

# Step 2: Helper to unwrap Document objects for easier output
def unwrap_document(answer):
    return answer | {"context": [{"metadata": r.metadata, "page_content": r.page_content} for r in answer["context"]]}

# Step 3: Build the chain
# Assume you have 'chat_model' defined (like a ChatDatabricks or OpenAI model)
# Assume your retriever is defined via 'get_retriever()'

question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
chain = create_retrieval_chain(get_retriever(), question_answer_chain) | RunnableLambda(unwrap_document)


In [0]:
import json

def pretty_display_result(result):
    print("\n" + "="*100)
    print(f"📋 Question:")
    print(f"{result.get('input', '')}")
    print("="*100)

    print("\n📚 Retrieved Contexts:")
    for idx, ctx in enumerate(result.get('context', []), start=1):
        source = ctx.get('metadata', {}).get('source', 'Unknown Source')
        text_preview = ctx.get('page_content', '').replace('\n', ' ').replace('  ', ' ').strip()
        print(f"\n🔹 Context #{idx}:")
        print(f"Source: {source}")
        print(f"Text Preview: {text_preview[:500]}...")  # limit to first 500 chars for readability

    print("\n" + "="*100)
    print(f"🧠 Final Answer:")
    print(f"{result.get('answer', '')}")
    print("="*100)


In [0]:
question = {"input": "story where two goats fight and fall into the river"}
answer = chain.invoke(question)
pretty_display_result(answer)

## Save the Model to Model Registry in UC

Now that our model is ready and evaluated, we can register it within our Unity Catalog schema. 

After registering the model, you can view the model and models in the **Catalog Explorer**.

In [0]:
from mlflow.models import infer_signature
import mlflow
import langchain

# set model registry to UC
mlflow.set_registry_uri("databricks-uc")
model_name = f"training_catalog.training_schema.rag_app_demo1"

with mlflow.start_run(run_name="rag_app_demo4") as run:
    signature = infer_signature(question, answer)
    model_info = mlflow.langchain.log_model(
        chain,
        loader_fn=get_retriever, 
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch",
        ],
        input_example=question,
        signature=signature
    )